# Building a Generation Model with End-to-End Training Loop

In [ ]:
# Import Modules
import torch
import torch.nn as nn
import math

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        seq_len = x.shape[0]
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)
        Q = Q.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        K = K.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)
        V = V.view(seq_len, self.num_heads, self.d_k).permute(1, 0, 2)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)

        # CAUSAL MASK — always applied here since there's no cache
        # scores shape: (num_heads, seq_len, seq_len)
        # mask shape:   (seq_len, seq_len) — broadcasts across heads automatically
        mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device),
            diagonal=1
        ).bool()
        scores = scores.masked_fill(mask, float('-inf'))

        attention_weights = torch.softmax(scores, dim=-1)
        head_outputs = attention_weights @ V
        head_outputs = head_outputs.permute(1, 0, 2).contiguous().view(seq_len, self.d_model)
        return self.W_O(head_outputs), attention_weights

In [ ]:
# Existing Feed Forward Network:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.layer1 = nn.Linear(d_model, d_ff)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))

In [ ]:
# Existing Transformer Block:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attention    = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.attention(self.norm1(x))
        x = x + attn_out
        ff_out = self.feed_forward(self.norm2(x))
        x = x + ff_out
        return x

In [ ]:
# Positional Encoding:
def positional_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            denom = 10000 ** (i / d_model)
            pe[pos, i]     = math.sin(pos / denom)
            pe[pos, i + 1] = math.cos(pos / denom)
    return pe

In [ ]:
class MiniLLM(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)

        # NEW: instead of ONE block, create a LIST of N blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])

        # Final layer norm before the output head (standard in GPT-style models)
        self.final_norm = nn.LayerNorm(d_model)

        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        seq_len = token_ids.shape[0]

        # Step 1: Token embeddings + positional encoding
        x = self.embedding(token_ids)
        x = x + positional_encoding(seq_len, x.shape[1])

        # Step 2: Pass through EACH block, one after another
        for block in self.blocks:
            x = block(x)

        # Step 3: Final normalization
        x = self.final_norm(x)

        # Step 4: Output head
        logits = self.output_head(x)

        return logits

In [ ]:
# text = """
# the quick brown fox jumps over the lazy dog
# artificial intelligence is transforming industries across the world
# machine learning models learn patterns from historical data
# deep learning networks contain multiple layers of neurons
# natural language processing enables communication between humans and machines
# computer vision systems analyze images and videos
# cloud platforms provide scalable infrastructure for applications
# software developers write maintainable and efficient code
# devops engineers automate deployment pipelines
# kubernetes orchestrates containerized applications at scale
# databases store and retrieve information efficiently
# cybersecurity protects systems against threats and attacks
# data scientists explore datasets to discover insights
# backend services expose apis for client applications
# frontend frameworks create interactive user interfaces
# microservices architectures break applications into smaller services
# distributed systems coordinate work across multiple machines
# version control systems track changes in source code
# testing frameworks ensure software quality and reliability
# automation improves productivity and reduces errors
# """

In [ ]:
# ── Training (same as before, just with num_layers) ────────────────────

text   = "the cat sat on the mat because it was tired"
tokens = text.lower().split()
vocab  = sorted(set(tokens))
word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for w, i in word_to_id.items()}
token_ids = torch.tensor([word_to_id[t] for t in tokens])

print("Number of tokens:", len(tokens))
print("Vocabulary size:", len(vocab))
print("\nVocabulary:")
print(vocab)
print("\nToken IDs:")
print(token_ids)

inputs  = token_ids[:-1]
targets = token_ids[1:]

# Config
vocab_size = len(vocab)
d_model    = 8
num_heads  = 2
d_ff       = 32
num_layers = 3            # ← NEW! 3 stacked transformer blocks

model     = MiniLLM(vocab_size, d_model, num_heads, d_ff, num_layers)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn   = nn.CrossEntropyLoss()

print(f"Model has {sum(p.numel() for p in model.parameters())} parameters")
print(f"Stack depth: {num_layers} transformer blocks\n")

for step in range(250):
    logits = model(inputs)
    loss   = loss_fn(logits, targets)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 20 == 0:
        print(f"Step {step:3d} | Loss: {loss.item():.4f}")

# Final predictions
print("\n=== Final Predictions ===")
with torch.no_grad():
    logits = model(inputs)
    predicted_ids = logits.argmax(dim=-1)

for i, (inp, pred, target) in enumerate(zip(inputs, predicted_ids, targets)):
    input_word  = id_to_word[inp.item()]
    pred_word   = id_to_word[pred.item()]
    target_word = id_to_word[target.item()]
    correct     = "✓" if pred.item() == target.item() else "✗"
    print(f"  [{correct}] Input: '{input_word}' → Predicted: '{pred_word}' (should be: '{target_word}')")

Number of tokens: 10
Vocabulary size: 9

Vocabulary:
['because', 'cat', 'it', 'mat', 'on', 'sat', 'the', 'tired', 'was']

Token IDs:
tensor([6, 1, 5, 4, 6, 3, 0, 2, 8, 7])
Model has 2689 parameters
Stack depth: 3 transformer blocks

Step   0 | Loss: 2.5034
Step  20 | Loss: 0.6581
Step  40 | Loss: 0.1574
Step  60 | Loss: 0.0441
Step  80 | Loss: 0.0217
Step 100 | Loss: 0.0141
Step 120 | Loss: 0.0103
Step 140 | Loss: 0.0079
Step 160 | Loss: 0.0063
Step 180 | Loss: 0.0052
Step 200 | Loss: 0.0044
Step 220 | Loss: 0.0037
Step 240 | Loss: 0.0032

=== Final Predictions ===
  [✓] Input: 'the' → Predicted: 'cat' (should be: 'cat')
  [✓] Input: 'cat' → Predicted: 'sat' (should be: 'sat')
  [✓] Input: 'sat' → Predicted: 'on' (should be: 'on')
  [✓] Input: 'on' → Predicted: 'the' (should be: 'the')
  [✓] Input: 'the' → Predicted: 'mat' (should be: 'mat')
  [✓] Input: 'mat' → Predicted: 'because' (should be: 'because')
  [✓] Input: 'because' → Predicted: 'it' (should be: 'it')
  [✓] Input: 'it' → Pr

In [ ]:
# ── Generation function ──────────────────────────────────────────────

def generate(
    model,
    prompt_text,
    word_to_id,
    id_to_word,
    max_new_tokens=15,
    temperature=1.0,
    do_sample=False,
):
    """
    Generate text from a trained MiniGPT model.

    Args:
        model:          trained MiniGPT
        prompt_text:    starting string (must use words from vocab)
        word_to_id:     vocab dict {word: id}
        id_to_word:     reverse vocab dict {id: word}
        max_new_tokens: how many tokens to generate
        temperature:    >0; lower = more focused, higher = more random
        do_sample:      if False, greedy (argmax). If True, sample.

    Returns:
        Generated text as a string.
    """
    model.eval()  # tells the model "we're in inference mode, not training"

    # Step 1: Tokenize the prompt
    prompt_tokens = prompt_text.split()
    token_ids = torch.tensor([word_to_id[t] for t in prompt_tokens])

    print(f"Starting prompt: {prompt_text!r} → tokens: {token_ids.tolist()}")

    # Step 2: Generation loop
    with torch.no_grad():  # no gradients needed — saves memory and time
        for step in range(max_new_tokens):

            # Forward pass on the current sequence
            logits = model(token_ids)              # shape: (seq_len, vocab_size)

            # Pull out only the LAST position's logits — that's what predicts next
            last_logits = logits[-1]               # shape: (vocab_size,)

            # Apply temperature (only matters when sampling)
            scaled_logits = last_logits / temperature

            # Convert to probabilities
            probs = torch.softmax(scaled_logits, dim=-1)

            # Pick the next token
            if do_sample:
                next_token_id = torch.multinomial(probs, num_samples=1).item()
            else:
                next_token_id = probs.argmax().item()

            # Show what happened at this step
            next_word = id_to_word[next_token_id]
            top_prob  = probs[next_token_id].item()
            print(f"  Step {step+1}: picked '{next_word}' (prob={top_prob:.4f})")

            # Append the new token to the sequence
            new_token_tensor = torch.tensor([next_token_id])
            token_ids = torch.cat([token_ids, new_token_tensor])

    # Step 3: Detokenize back to text
    generated_words = [id_to_word[t.item()] for t in token_ids]
    return " ".join(generated_words)


### Code Explanation
`model.eval()` — tells the model it's not training anymore. Some layers (dropout, batch norm) behave differently in train vs eval mode. Our MiniGPT doesn't use those, but it's a good habit. It's the inference-time counterpart to `model.train()`.

`with torch.no_grad()`: — wraps the whole generation loop. PyTorch normally builds a computation graph during the forward pass so it can compute gradients on backward. During generation we don't need gradients — we're not learning anything, just producing output. Skipping the graph saves memory and runs faster. Critical habit for inference code.

`logits = model(token_ids)` — same forward pass you've been doing during training. Output shape (seq_len, vocab_size).

`last_logits = logits[-1]` — pull out just the final row. This is the prediction for what comes AFTER the last token. We don't care about earlier positions during generation (they predict tokens we already have).

`scaled_logits = last_logits / temperature` — applies temperature BEFORE softmax. Lower T sharpens; higher T flattens. When temperature=1.0, this is a no-op (logits / 1 = logits).

`probs = torch.softmax(scaled_logits, dim=-1)` — convert logits to probabilities. After this, all values are in [0, 1] and sum to 1.0.

### Greedy vs sampling branch:

`probs.argmax().item()` — argmax returns a tensor with the index of the highest value. `.item()` converts that single-element tensor into a regular Python int. Always picks the most likely token.

`torch.multinomial(probs, num_samples=1).item()` — treats probs as a weighted die and rolls it once. Returns a tensor of shape (1,), which `.item()` converts to int.

`torch.cat([token_ids, new_token_tensor])` — concatenates the new token onto the existing sequence. Now next iteration's forward pass will run on a sequence one token longer. This is the "loop body" of autoregressive generation.

The detokenize step at the end — convert IDs back to words and join them. The standard "tokens → text" reversal.

What you'll observe:

- Mode 1 (Greedy): Output should be "i love code and i love ai" exactly. Every time. Deterministic. The probabilities printed at each step should be near 1.0 — the model is confident.
- Mode 2 (T=1.0, sampling): Almost identical output to greedy, because the model's probabilities are so peaked. You might occasionally see a variation, but rarely.
- Mode 3 (T=2.0): Starts to deviate. The flattened distribution means the model can pick "wrong" tokens. You might see "i love code and ai love ai" or similar nonsense.
- Mode 4 (T=5.0): Almost completely random — at this temperature the model's signal is drowned out by the random-noise floor, and you'll get strings that look like the model is babbling.

In [ ]:
print("=" * 60)
print("MODE 1: Greedy (deterministic)")
print("=" * 60)
output = generate(
    model, "the cat",
    word_to_id, id_to_word,
    max_new_tokens=8,
    do_sample=False,
)
print(f"\nFinal output: {output!r}\n")

MODE 1: Greedy (deterministic)
Starting prompt: 'the cat' → tokens: [6, 1]
  Step 1: picked 'sat' (prob=0.9974)
  Step 2: picked 'on' (prob=0.9971)
  Step 3: picked 'the' (prob=0.9976)
  Step 4: picked 'mat' (prob=0.9964)
  Step 5: picked 'because' (prob=0.9969)
  Step 6: picked 'it' (prob=0.9969)
  Step 7: picked 'was' (prob=0.9973)
  Step 8: picked 'tired' (prob=0.9961)

Final output: 'the cat sat on the mat because it was tired'



In [ ]:
print("=" * 60)
print("MODE 2: Sampling with T=1.0 (raw model probabilities)")
print("=" * 60)
output = generate(
    model, "the cat",
    word_to_id, id_to_word,
    max_new_tokens=8,
    do_sample=True,
    temperature=1.0,
)
print(f"\nFinal output: {output!r}\n")

MODE 2: Sampling with T=1.0 (raw model probabilities)
Starting prompt: 'the cat' → tokens: [6, 1]
  Step 1: picked 'sat' (prob=0.9974)
  Step 2: picked 'on' (prob=0.9971)
  Step 3: picked 'the' (prob=0.9976)
  Step 4: picked 'mat' (prob=0.9964)
  Step 5: picked 'because' (prob=0.9969)
  Step 6: picked 'it' (prob=0.9969)
  Step 7: picked 'was' (prob=0.9973)
  Step 8: picked 'tired' (prob=0.9961)

Final output: 'the cat sat on the mat because it was tired'



In [ ]:
print("=" * 60)
print("MODE 3: Sampling with T=2.0 (more random)")
print("=" * 60)
output = generate(
    model, "the cat",
    word_to_id, id_to_word,
    max_new_tokens=8,
    do_sample=True,
    temperature=2.0,
)
print(f"\nFinal output: {output!r}\n")

MODE 3: Sampling with T=2.0 (more random)
Starting prompt: 'the cat' → tokens: [6, 1]
  Step 1: picked 'sat' (prob=0.8928)
  Step 2: picked 'on' (prob=0.8759)
  Step 3: picked 'the' (prob=0.8903)
  Step 4: picked 'mat' (prob=0.8735)
  Step 5: picked 'because' (prob=0.8846)
  Step 6: picked 'it' (prob=0.8783)
  Step 7: picked 'was' (prob=0.8774)
  Step 8: picked 'tired' (prob=0.8586)

Final output: 'the cat sat on the mat because it was tired'



In [ ]:
print("=" * 60)
print("MODE 4: Sampling with T=5.0 (chaos)")
print("=" * 60)
output = generate(
    model, "the cat",
    word_to_id, id_to_word,
    max_new_tokens=8  ,
    do_sample=True,
    temperature=5.0,
)
print(f"\nFinal output: {output!r}\n")

MODE 4: Sampling with T=5.0 (chaos)
Starting prompt: 'the cat' → tokens: [6, 1]
  Step 1: picked 'sat' (prob=0.4238)
  Step 2: picked 'sat' (prob=0.0884)
  Step 3: picked 'the' (prob=0.1505)
  Step 4: picked 'cat' (prob=0.0880)
  Step 5: picked 'sat' (prob=0.3022)
  Step 6: picked 'the' (prob=0.0549)
  Step 7: picked 'cat' (prob=0.2839)
  Step 8: picked 'on' (prob=0.0924)

Final output: 'the cat sat sat the cat sat the cat on'

